In [7]:
from pathlib import Path

import numpy as np
import plotly.express as px
import polars as pl
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader

from tarp.cli.logging import Console
from tarp.services.datasets.classification.multilabel import (
    MultiLabelClassificationDataset,
)
from tarp.services.datasources.sequence import TabularSequenceSource, FastaSliceSource
from tarp.services.evaluation.classification.multilabel import MultiLabelMetrics
from tarp.services.tokenizers.pretrained.dnabert2 import Dnabert2Tokenizer
from tarp.model.backbone.untrained.transformer import TransformerEncoder
from tarp.config import TransformerConfig
from tarp.model.finetuning.classification import ClassificationModel

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = Dnabert2Tokenizer()
label_columns = pl.read_csv(Path("../temp/data/cache/labels.csv")).to_series().to_list()
pretrained_encoder = encoder = TransformerEncoder(
    vocabulary_size=tokenizer.vocab_size,
    embedding_dimension=TransformerConfig.embedding_dimension,
    feedforward_dimension=TransformerConfig.feedforward_dimension,
    padding_id=tokenizer.pad_token_id,
    number_of_layers=TransformerConfig.number_of_layers,
    number_of_heads=TransformerConfig.number_of_heads,
    dropout=TransformerConfig.dropout,
).to(device)
model = ClassificationModel(
    encoder=pretrained_encoder,
    number_of_classes=len(label_columns),
).to(device)
# Load pretrained weights
model.load_state_dict(torch.load(Path("../temp/checkpoints/20251216_212224/classification_model_full_20251216_212224.pt")))
# Set to evaluation mode
model.eval()
Console.info("Model and classification head initialized")

[INFO]	2025-12-17 10:35:57,329 - Model and classification head initialized


In [9]:
print(device)

cuda


In [10]:
multilabel_classification_valid = MultiLabelClassificationDataset(
    (
        TabularSequenceSource(
            source=Path("../temp/data/processed/card_amr.valid.parquet"),
        )
        # + FastaSliceSource(
        #     directory=Path("temp/data/external/sequences/nucleotide/"),
        #     metadata=Path("temp/data/processed/fine_tuning.valid.parquet"),
        #     key_column="protein_accession.version",
        #     start_column="start",
        #     end_column="end",
        #     sequence_column="dna_sequence",
        # )
    ),
    tokenizer=tokenizer,
    sequence_column="protein_sequence",
    label_columns=label_columns,
    maximum_sequence_length=200,
)
valid_loader = DataLoader(
    multilabel_classification_valid,
    batch_size=16,
    shuffle=False,
    num_workers=4,
)

In [11]:
# Predict on validation set to obtain probabilities
all_probs = []
all_labels = []
model.eval()
with torch.no_grad():
    for batch in tqdm(valid_loader, desc="Validating", unit="batch"):
        input_ids = batch["sequence"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()
        outputs = model(input_ids, attention_mask=attention_mask)
        all_probs.append(outputs.cpu().numpy())
        all_labels.append(labels)

all_probs = np.vstack(all_probs)
all_labels = np.vstack(all_labels)

Validating: 100%|██████████| 78/78 [00:22<00:00,  3.53batch/s]


In [12]:
thresholds = np.linspace(0.0, 1.0, 101)
rows = []

for thr in thresholds:
    metrics = MultiLabelMetrics(threshold=thr).compute(
        torch.as_tensor(all_probs),
        torch.as_tensor(all_labels),
    )
    rows.append({"threshold": thr, **{k: float(v) for k, v in metrics.items()}})

[WARN]	2025-12-17 10:36:19,474 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10:36:19,659 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10:36:19,820 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10:36:19,971 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10:36:20,112 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10:36:20,274 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10:36:20,433 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10:36:20,612 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10:36:20,773 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10:36:20,961 - Skipping 16 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-17 10

In [13]:
df = pl.DataFrame(rows)

metric_cols = [c for c in df.columns if c != "threshold"]

df_long = df.unpivot(
    on=metric_cols, index="threshold", variable_name="metric", value_name="value"
)

# Plotly expects pandas df
fig = px.line(
    df_long.to_pandas(),
    x="threshold",
    y="value",
    color="metric",
    title="Multi-Label Metrics Across Thresholds",
    markers=True,
)

target_metric = "f1"

best_threshold = (
    df.sort(target_metric, descending=True)
      .select("threshold")
      .head(1)
      .item()
)

best_metric_value = (
    df.sort(target_metric, descending=True)
      .select(target_metric)
      .head(1)
      .item()
)

Console.info(
    f"Best threshold: {best_threshold:.2f} with {target_metric}: {best_metric_value:.4f}"
)
fig.show()

[INFO]	2025-12-17 10:36:34,873 - Best threshold: 0.87 with f1: 0.2852
